# 03 — Data Preparation Pipeline
## STP Natality — Weight Analysis of Newborns

> **Phase 2.4** | Author: STP Natality Team | Last updated: June 2026

This notebook executes the **full end-to-end preprocessing and splitting pipeline** on the complete dataset (~27M rows), producing clean, version-controlled Parquet splits in `data/processed/`.

### Pipeline Steps
1. Setup & Configuration
2. Data Ingestion (`src.data.ingest`)
3. Preprocessing (`src.data.preprocess`)
4. Feature Engineering (`src.data.feature_engineering`)
5. Temporal Train/Val/Test Split (`src.data.split`)
6. Save Processed Splits to Parquet
7. Pipeline Summary & Validation

---
## 1. Setup & Configuration

In [ ]:
import logging
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Project root ───────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── Logging ────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(name)s | %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('pipeline')

# ── Project modules ────────────────────────────────────────────────────────────
from src.data.ingest           import load_all_files, get_data_summary, NATALITY_FILES
from src.data.preprocess       import run_preprocessing
from src.data.feature_engineering import engineer_features
from src.data.split            import temporal_split, save_splits, get_feature_columns, get_X_y

plt.rcParams.update({'figure.dpi': 110, 'figure.facecolor': 'white'})

print(f'Python {sys.version}')
print(f'pandas {pd.__version__} | numpy {np.__version__}')
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# ── Pipeline configuration ─────────────────────────────────────────────────────
# Set NROWS_PER_FILE = None for the full ~27M-row run.
# Set to an integer (e.g. 200_000) for a fast dev/test run.
NROWS_PER_FILE = None           # None = FULL dataset
RAW_DATA_DIR   = PROJECT_ROOT / 'newborn_data'
OUT_DIR        = PROJECT_ROOT / 'data' / 'processed'

logger.info('Pipeline configuration:')
logger.info('  Raw data dir : %s', RAW_DATA_DIR)
logger.info('  Output dir   : %s', OUT_DIR)
logger.info('  Rows per file: %s', NROWS_PER_FILE if NROWS_PER_FILE else 'ALL')

---
## 2. Data Ingestion

In [ ]:
t0 = time.time()

df_raw = load_all_files(
    data_dir=RAW_DATA_DIR,
    nrows_per_file=NROWS_PER_FILE,
)

ingestion_time = time.time() - t0
logger.info('Ingestion complete in %.1fs', ingestion_time)

summary = get_data_summary(df_raw)
print(f'\n=== Ingestion Summary ===')
print(f'  Rows:       {summary["n_rows"]:>12,}')
print(f'  Columns:    {summary["n_cols"]:>12}')
print(f'  Memory:     {summary["memory_mb"]:>11.1f} MB')
print(f'  Year range: {summary["year_range"]["min"]} – {summary["year_range"]["max"]}')
print(f'  Files:      {summary["files_included"]}')

In [ ]:
# ── Row counts per source file ─────────────────────────────────────────────────
file_counts = df_raw.groupby('source_file').size().reset_index(name='row_count')
file_counts['pct'] = (file_counts['row_count'] / len(df_raw) * 100).round(1)
display(file_counts)

# Verify all 5 files loaded
assert len(file_counts) == 5, f'Expected 5 files, got {len(file_counts)}'
print('\n✅ All 5 natality files loaded successfully.')

---
## 3. Preprocessing

Steps applied:
1. Drop leakage columns (`apgar_1min`, `apgar_5min`, `day`)
2. Cast boolean strings to Int8
3. Replace sentinel values with NaN (`father_age=99`, `gestation_weeks=99`, `lmp=99/9999`)
4. Clip outliers to clinical ranges and flag weight outliers
5. Convert target: `weight_pounds → weight_grams`
6. Drop rows with null or clinically invalid birth weight

In [ ]:
t1 = time.time()
rows_before = len(df_raw)

df_clean = run_preprocessing(df_raw)

preprocessing_time = time.time() - t1
rows_removed = rows_before - len(df_clean)

print(f'\n=== Preprocessing Summary ===')
print(f'  Input rows:   {rows_before:>12,}')
print(f'  Output rows:  {len(df_clean):>12,}')
print(f'  Rows removed: {rows_removed:>12,}  ({rows_removed/rows_before*100:.2f}%)')
print(f'  Columns:      {len(df_clean.columns):>12}')
print(f'  Time:         {preprocessing_time:>11.1f}s')
print(f'  Memory:       {df_clean.memory_usage(deep=True).sum()/1e6:>11.1f} MB')

In [ ]:
# ── Post-preprocessing statistics ─────────────────────────────────────────────
print('Missing values after preprocessing (% of rows):')
missing_clean = df_clean.isnull().mean().mul(100).sort_values(ascending=False)
display(missing_clean[missing_clean > 0].to_frame('missing_%').round(2))

print('\nTarget variable (weight_grams) summary:')
display(df_clean['weight_grams'].describe().round(1).to_frame('weight_grams').T)

---
## 4. Feature Engineering

New features added:
| Feature | Description |
|---------|-------------|
| `weight_gain_kg` | Maternal weight gain in kilograms |
| `parity` | Total prior births (born_alive_alive + born_alive_dead + born_dead) |
| `is_multiple_birth` | 1 if plurality > 1 (twins, triplets, etc.) |
| `lmp_known` | 1 if last menstrual period date is known |
| `birth_month_sin/cos` | Cyclical encoding of birth month |
| `mother_age_group` | Ordinal age group: 0=teen, 1=20s, 2=30s, 3=40+ |
| `gestation_preterm` | 1 if gestation_weeks < 37 |
| `gestation_post_term` | 1 if gestation_weeks > 41 |

In [ ]:
t2 = time.time()
cols_before = df_clean.columns.tolist()

df_feat = engineer_features(df_clean)

feat_time = time.time() - t2
new_cols = [c for c in df_feat.columns if c not in cols_before]

print(f'\n=== Feature Engineering Summary ===')
print(f'  Input columns:  {len(cols_before):>8}')
print(f'  Output columns: {len(df_feat.columns):>8}')
print(f'  New features:   {new_cols}')
print(f'  Time:           {feat_time:.2f}s')

In [ ]:
# ── Engineered feature quick check ────────────────────────────────────────────
eng_stats = df_feat[new_cols].describe().round(3)
display(eng_stats)

---
## 5. Temporal Train / Validation / Test Split

**Strategy:** Year-based temporal split (no random split to avoid data leakage across time).

| Split | Years | Rationale |
|-------|-------|-----------|
| **Train** | < 2010 | Historical data (~1986–2009) |
| **Validation** | 2010–2015 | For hyperparameter tuning |
| **Test** | ≥ 2016 | Final evaluation — never touched during training |

In [ ]:
t3 = time.time()

train, val, test = temporal_split(df_feat)

split_time = time.time() - t3
total_rows = len(df_feat)

print('\n=== Split Summary ===')
for name, split in [('Train', train), ('Val', val), ('Test', test)]:
    yr_min = int(split['year_fix'].min())
    yr_max = int(split['year_fix'].max())
    pct    = len(split) / total_rows * 100
    lbw    = (split['weight_grams'] < 2500).mean() * 100
    print(f'  {name:<6}: {len(split):>10,} rows  ({pct:.1f}%)  years {yr_min}–{yr_max}  LBW: {lbw:.2f}%')

print(f'  Time: {split_time:.2f}s')

In [ ]:
# ── Feature / target columns ───────────────────────────────────────────────────
feature_cols = get_feature_columns(df_feat)
print(f'Model input features ({len(feature_cols)} total):')
for col in sorted(feature_cols):
    dtype = df_feat[col].dtype
    missing_pct = df_feat[col].isna().mean() * 100
    print(f'  {col:<30} {str(dtype):<12} missing: {missing_pct:.1f}%')

In [ ]:
# ── Visual split distribution check ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Split proportions
ax = axes[0]
split_sizes = pd.Series({'Train': len(train), 'Val': len(val), 'Test': len(test)})
colors = ['#2196F3', '#FF9800', '#4CAF50']
ax.bar(split_sizes.index, split_sizes.values, color=colors, alpha=0.85)
ax.set_title('Split Sizes')
ax.set_ylabel('Number of Records')
for i, (k, v) in enumerate(split_sizes.items()):
    ax.text(i, v + 0.5, f'{v:,}\n({v/total_rows*100:.1f}%)', ha='center', va='bottom', fontsize=9)

# Weight distributions per split
ax = axes[1]
for name, split, color in [('Train', train, '#2196F3'), ('Val', val, '#FF9800'), ('Test', test, '#4CAF50')]:
    split['weight_grams'].dropna().plot.kde(ax=ax, label=name, color=color, linewidth=2)
ax.axvline(2500, color='gray', linestyle='--', linewidth=1, label='LBW')
ax.set_xlabel('Birth Weight (g)')
ax.set_title('Weight Distribution per Split')
ax.legend()

plt.suptitle('Temporal Split Validation', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Save Processed Splits to Parquet

In [ ]:
t4 = time.time()

save_splits(train, val, test, out_dir=OUT_DIR)

save_time = time.time() - t4

print(f'\n=== Save Summary ===')
for fname in ['train.parquet', 'val.parquet', 'test.parquet']:
    p = OUT_DIR / fname
    if p.exists():
        mb = p.stat().st_size / 1e6
        print(f'  {fname:<20}: {mb:.1f} MB')

print(f'  Save time: {save_time:.1f}s')
print(f'  Output directory: {OUT_DIR}')

---
## 7. Pipeline Summary & Validation

In [ ]:
# ── Reload and verify the saved splits ────────────────────────────────────────
from src.data.split import load_splits

train_check, val_check, test_check = load_splits(data_dir=OUT_DIR)

print('=== Verification: Reloaded splits ===')
for name, df_check, df_orig in [
    ('Train', train_check, train),
    ('Val',   val_check,   val),
    ('Test',  test_check,  test),
]:
    rows_match = len(df_check) == len(df_orig)
    cols_match = list(df_check.columns) == list(df_orig.columns)
    print(f'  {name}: rows {len(df_check):,} (match={rows_match}), cols {len(df_check.columns)} (match={cols_match})')

print('\n✅ Splits saved and verified.')

In [ ]:
# ── Final pipeline timing summary ─────────────────────────────────────────────
total_time = ingestion_time + preprocessing_time + feat_time + split_time + save_time

print('\n' + '=' * 50)
print('PIPELINE EXECUTION SUMMARY')
print('=' * 50)
print(f'  Ingestion:          {ingestion_time:>7.1f}s')
print(f'  Preprocessing:      {preprocessing_time:>7.1f}s')
print(f'  Feature Engineering:{feat_time:>7.1f}s')
print(f'  Split:              {split_time:>7.1f}s')
print(f'  Save:               {save_time:>7.1f}s')
print(f'  ─────────────────────────────')
print(f'  TOTAL:              {total_time:>7.1f}s  ({total_time/60:.1f} min)')
print('=' * 50)
print(f'\nOutput: {OUT_DIR}')
print(f'  train.parquet : {len(train):>12,} rows')
print(f'  val.parquet   : {len(val):>12,} rows')
print(f'  test.parquet  : {len(test):>12,} rows')
print(f'  Features      : {len(feature_cols):>12}')
print(f'  Target        : weight_grams (birth weight in grams)')

In [ ]:
# ── Post-pipeline data quality checks ─────────────────────────────────────────
print('=== Data Quality Assertions ===')

# 1. No null target in any split
for name, split in [('train', train), ('val', val), ('test', test)]:
    null_targets = split['weight_grams'].isna().sum()
    status = '✅' if null_targets == 0 else '❌'
    print(f'  {status} No null weight_grams in {name}: {null_targets} nulls')

# 2. Weight grams in valid clinical range [300, 7000]
all_weights = pd.concat([train['weight_grams'], val['weight_grams'], test['weight_grams']])
out_of_range = ((all_weights < 300) | (all_weights > 7000)).sum()
status = '✅' if out_of_range == 0 else '❌'
print(f'  {status} All weight_grams in [300, 7000]g: {out_of_range} violations')

# 3. No temporal overlap between splits
train_max = train['year_fix'].max()
val_min   = val['year_fix'].min()
val_max   = val['year_fix'].max()
test_min  = test['year_fix'].min()
no_overlap = (train_max < val_min) and (val_max < test_min)
status = '✅' if no_overlap else '❌'
print(f'  {status} No temporal overlap between splits (train_max={train_max}, val_min={val_min}, val_max={val_max}, test_min={test_min})')

# 4. No leakage columns in feature set
leakage = [c for c in ['apgar_1min', 'apgar_5min'] if c in feature_cols]
status = '✅' if not leakage else '❌'
print(f'  {status} No leakage columns in feature set: {leakage if leakage else "none"}')

print('\nAll quality checks complete.')